# Dual Cost–Constraint projection (DCC): overview and toy example

Notes on Hu & Haddadin, *Optimal Control without Optimization*, arXiv:2502.09322v3 (28 Jul 2026),
[publication/Hu_and_Haddadin_Optimal Control without Optimization_2026.pdf](publication/).
The main text gives the compact forms (1)–(4). The details are in the Supplementary Information (SI),
which starts on PDF page 35. Equation numbers (S·) below refer to the SI.

Contents:

1. Problem class: equations (1a), (1b), (2), (3)
2. The DCC control law: equation (4) and its expansion (S4)
3. Key features and assumptions of the problem class
4. What DCC is, in familiar terms
5. Toy example: two-rotor thrust allocation (a 2-state version of Example R1)
6. Experiments: one Newton step per sample, gain $K_\mathrm{x}$, penalty $p_2$, clipped inputs
7. Other candidate toy systems
8. Summary of limitations, with page references

## 1. Problem class

### System, eq. (1) / (S1)

Nonlinear, control-affine system with time-varying output map:

$$
\dot{\boldsymbol{x}} = \boldsymbol{f}_\mathrm{A}(\boldsymbol{x}) + \boldsymbol{B}(\boldsymbol{x})\,\boldsymbol{u}, \tag{1a}
$$

$$
\boldsymbol{y} = \boldsymbol{h}(t, \boldsymbol{x}), \tag{1b}
$$

with $\boldsymbol{x} \in \mathbb{R}^{d_\mathrm{x}}$, $\boldsymbol{u} \in \mathbb{R}^{d_\mathrm{x}}$ (note: **same dimension as the state**),
$\boldsymbol{y} \in \mathbb{R}^{d_\mathrm{y}}$ with $d_\mathrm{y} < d_\mathrm{x}$, and $\boldsymbol{B}(\boldsymbol{x})$ square and invertible for all $\boldsymbol{x}$.

Because $d_\mathrm{y} < d_\mathrm{x}$ and $\boldsymbol{h}$ is surjective in $\boldsymbol{x}$, there are infinitely many states that give $\boldsymbol{y} = \boldsymbol{y}_\mathrm{ref}$.
The cost and inequality constraints pick the "best" one among them. This is a **redundancy resolution** or
**control allocation** setting.

### Optimality principle: static problem, eq. (2) / (S2)

At every time instant $t \ge 0$:

$$
\begin{aligned}
\min_{\boldsymbol{\chi}} \quad & \sigma(t, \boldsymbol{\chi}) && \text{(2a)} \\
\text{s.t.} \quad & \boldsymbol{0} = \boldsymbol{h}(t, \boldsymbol{\chi}) - \boldsymbol{y}_\mathrm{ref}(t) && \text{(2b)} \\
& \boldsymbol{0} \geq \boldsymbol{g}(t, \boldsymbol{\chi}) := \boldsymbol{G}(t)\,\boldsymbol{\chi} + \boldsymbol{c}(t) && \text{(2c)}
\end{aligned}
$$

### Optimality principle: optimal tracking problem, eq. (3) / (S3)

Over the period $[0, t_\mathrm{f}]$:

$$
\begin{aligned}
\min_{\boldsymbol{\phi}} \quad & \int_0^{t_\mathrm{f}} \sigma\big(t, \boldsymbol{\phi}(t)\big)\,\mathrm{d}t && \text{(3a)} \\
\text{s.t.} \quad & \boldsymbol{0} = \boldsymbol{h}\big(t, \boldsymbol{\phi}(t)\big) - \boldsymbol{y}_\mathrm{ref}(t) \quad \forall t \in (0, t_\mathrm{f}] && \text{(3b)} \\
& \boldsymbol{0} \geq \boldsymbol{g}\big(t, \boldsymbol{\phi}(t)\big) \quad \forall t \in (0, t_\mathrm{f}] && \text{(3c)} \\
& \boldsymbol{x}(0) = \boldsymbol{\phi}(0) && \text{(3d)}
\end{aligned}
$$

The state is called *optimal* if $\boldsymbol{x}(t) = \boldsymbol{\chi}^\star(t)$ or $\boldsymbol{x}(t) = \boldsymbol{\phi}^\star(t)$ (the paper writes $\boldsymbol{\chi}^{\oplus\otimes}$ for a local minimizer).

> **Note.** Problem (3) contains **no dynamics constraint** linking $\boldsymbol{\phi}$ to $\boldsymbol{u}$ and **no input cost**.
> The integrand depends only on $\boldsymbol{\phi}(t)$ at the same instant, so (3) decouples into a sequence of independent
> static problems (2). The SI proves this as Lemma 13: $\boldsymbol{\phi}^\star(n\,\delta t) = \boldsymbol{\chi}^\star(n\,\delta t)$.
> "Optimal control" here therefore means *tracking the time-varying solution of a static optimization problem*,
> not the usual dynamic trade-off of Pontryagin, LQR or MPC.

## 2. The DCC control law

### Compact form, eq. (4) / (S4a)

$$
\boldsymbol{u} = \boldsymbol{b}(\boldsymbol{x}) + \boldsymbol{K}(\boldsymbol{x})
\begin{bmatrix} \mathcal{P}\{\boldsymbol{h}\}(t, \boldsymbol{x}) & \mathcal{N}\{\boldsymbol{h}\}(t, \boldsymbol{x}) \end{bmatrix}
\begin{bmatrix} \boldsymbol{e}_\mathrm{y} \\ \boldsymbol{e}_{\sigma,g} \end{bmatrix} \tag{4}
$$

### Components, eqs. (S4b)–(S4g)

$$
\begin{aligned}
\boldsymbol{b}(\boldsymbol{x}) &:= -\boldsymbol{B}(\boldsymbol{x})^{-1}\boldsymbol{f}_\mathrm{A}(\boldsymbol{x})
&& \text{compensator: cancels the drift (feedback linearization)} \\
\boldsymbol{K}(\boldsymbol{x}) &:= \boldsymbol{B}(\boldsymbol{x})^{-1} K_\mathrm{x}
&& \text{gain scheduling, } K_\mathrm{x} > 0 \text{ a scalar design gain} \\
\mathcal{P}\{\boldsymbol{h}\} &:= \boldsymbol{H}^{\#}_{\boldsymbol{R}}
= \boldsymbol{R}^{-1}\boldsymbol{H}^\mathsf{T}\big(\boldsymbol{H}\boldsymbol{R}^{-1}\boldsymbol{H}^\mathsf{T}\big)^{-1}
&& \boldsymbol{R}\text{-weighted pseudoinverse of the output Jacobian} \\
\mathcal{N}\{\boldsymbol{h}\} &:= \boldsymbol{I} - \boldsymbol{H}^{\#}_{\boldsymbol{R}}\boldsymbol{H}
&& \text{associated null-space projector} \\
\boldsymbol{e}_\mathrm{y} &:= \boldsymbol{y}_\mathrm{ref} - \boldsymbol{h}(t, \boldsymbol{x})
&& \text{output (equality-constraint) error} \\
\boldsymbol{e}_{\sigma,g} &:= -\boldsymbol{R}(t, \boldsymbol{x})^{-1}\boldsymbol{r}(t, \boldsymbol{x})
&& \text{Newton step on the augmented cost}
\end{aligned}
$$

where $\boldsymbol{H} = \partial\boldsymbol{h}/\partial\boldsymbol{x}$, and $\boldsymbol{r}$ and $\boldsymbol{R}$ are the gradient and Hessian of the **augmented cost** (S6):

$$
\rho(t, \boldsymbol{\chi}) := \sigma(t, \boldsymbol{\chi}) + \tfrac{1}{2}\,\boldsymbol{\beta}\big(\boldsymbol{g}(t,\boldsymbol{\chi})\big)^\mathsf{T}\boldsymbol{\beta}\big(\boldsymbol{g}(t,\boldsymbol{\chi})\big),
\qquad
\beta(s) = \frac{p_2}{p_1}\ln\big(1 + e^{p_1 s}\big). \tag{S6, S7}
$$

### The "barrier" is really an exterior penalty, eq. (S70)

The paper then lets $p_1 \to \infty$. The softplus becomes $p_2\max(0, s)$, and $\tfrac12\beta^2$ becomes the **quadratic exterior penalty**
$\tfrac12 p_2^2 \max(0, s)^2$. With $\breve{\boldsymbol{G}}, \breve{\boldsymbol{g}}$ the rows of the *currently violated* constraints:

$$
\boldsymbol{r} = \begin{cases} \boldsymbol{q} & \boldsymbol{g} \le \boldsymbol{0} \\ \boldsymbol{q} + p_2^2\,\breve{\boldsymbol{G}}^\mathsf{T}\breve{\boldsymbol{g}} & \text{otherwise} \end{cases}
\qquad
\boldsymbol{R} = \begin{cases} \boldsymbol{Q} & \boldsymbol{g} \le \boldsymbol{0} \\ \boldsymbol{Q} + p_2^2\,\breve{\boldsymbol{G}}^\mathsf{T}\breve{\boldsymbol{G}} & \text{otherwise} \end{cases}
\tag{S70}
$$

with $\boldsymbol{q} = \nabla_{\boldsymbol{x}}\sigma$ and $\boldsymbol{Q} = \nabla^2_{\boldsymbol{x}}\sigma$. Despite the "interior-point" language in the derivation,
the inequality constraints are enforced **from outside**: they can be violated slightly, by roughly $O(1/p_2^2)$. The paper recommends $p_2$ such that
$\|p_2^2\breve{\boldsymbol{G}}^\mathsf{T}\breve{\boldsymbol{G}}\| \gtrsim 1000\,\|\boldsymbol{Q}\|$ (S72).

### Closed loop, eq. (S5)

Substituting (4) into (1a), the drift and $\boldsymbol{B}$ cancel exactly:

$$
\dot{\boldsymbol{x}} = K_\mathrm{x}\Big[\underbrace{\boldsymbol{H}^{\#}_{\boldsymbol{R}}\big(\boldsymbol{y}_\mathrm{ref} - \boldsymbol{h}(t,\boldsymbol{x})\big)}_{\text{I: output tracking}}
+ \underbrace{\big(\boldsymbol{I} - \boldsymbol{H}^{\#}_{\boldsymbol{R}}\boldsymbol{H}\big)\big(-\boldsymbol{R}^{-1}\boldsymbol{r}\big)}_{\text{II: Newton step projected onto the null space}}\Big]
=: K_\mathrm{x}\,\boldsymbol{\eta}(t, \boldsymbol{x}, \boldsymbol{y}_\mathrm{ref}) \tag{S5}
$$

Read as an iteration, $\boldsymbol{\chi}_{k+1} = \boldsymbol{\chi}_k + \boldsymbol{\eta}(t, \boldsymbol{\chi}_k, \boldsymbol{y}_\mathrm{ref})$, this is the paper's "improved interior-point iteration" (S17).
In Newton terms it is a quasi-Newton (chord) step on the KKT system of (S8), with the multiplier-curvature term dropped from the Hessian.
DCC is the continuous-time version of that iteration, scaled by $K_\mathrm{x}$ and run *on the plant itself*.

Claimed behaviour (Theorem 1): locally, $\|\boldsymbol{x}(t) - \boldsymbol{\chi}^\star\| \le \|\boldsymbol{x}(0) - \boldsymbol{\chi}^\star\|\,e^{-\alpha K_\mathrm{x} t}$ for a constant minimizer,
and a tracking error bounded by $D_\chi / (\alpha K_\mathrm{x})$ when $\|\dot{\boldsymbol{\chi}}^\star\| \le D_\chi$, with $\alpha \in (0, 0.5)$.

## 3. Key features and assumptions of the problem class

Collected from SI Section 1 (Assumptions 1), Theorem 1 and the Discussion.

| Feature | What the paper requires | Comment |
|---|---|---|
| **Control-affine dynamics** | $\dot{\boldsymbol x} = \boldsymbol f_\mathrm{A}(\boldsymbol x) + \boldsymbol B(\boldsymbol x)\boldsymbol u$ | Standard nonlinear control form. $\boldsymbol f_\mathrm{A}$ and $\boldsymbol B$ are time-invariant; only $\boldsymbol h, \sigma, \boldsymbol g$ vary with $t$. |
| **Full actuation** | $\boldsymbol B(\boldsymbol x)$ square and **invertible for all** $\boldsymbol x$, so $\dim \boldsymbol u = \dim \boldsymbol x$ | The decisive restriction. The controller can set $\dot{\boldsymbol x}$ to anything, so the plant is effectively $\dot{\boldsymbol x} = \boldsymbol v$ after feedback linearization. Most process systems (CSTRs, columns, networks) are underactuated. The paper's own Discussion admits control-affinity with $d_u = d_x$ fails for "many networked systems". |
| **Existence and uniqueness** | $\boldsymbol f_\mathrm{A}, \boldsymbol B$ give unique trajectories (e.g. locally Lipschitz) | The usual condition. |
| **Smooth output map** | $\boldsymbol h(t, \cdot)$ is $C^2$ in $\boldsymbol x$, continuous in $t$, **surjective**, $d_\mathrm{y} < d_\mathrm{x}$ | Redundancy is what leaves room for optimization. $\boldsymbol H$ must have full row rank along the path, or the pseudoinverse is singular. |
| **Strongly convex cost** | $\sigma(t, \cdot)$ $C^2$ and **strongly convex** (Assumption 1.1) | The title and abstract say "pseudoconvex", but the theory assumes strong convexity of $\sigma$. Non-convexity enters only through a nonlinear equality $\boldsymbol h$. The Hessian must be cheap to evaluate. |
| **Linear inequality constraints** | $\boldsymbol g = \boldsymbol G(t)\boldsymbol x + \boldsymbol c(t)$ (affine in $\boldsymbol x$, may vary with time) | Handled by an exterior quadratic penalty, so they hold only approximately. Nonlinear inequalities are future work. |
| **Constraints on the state only** | Cost and constraints are on $\boldsymbol x$ (the "decision"), not on $\boldsymbol u$ | No input limits or rate limits. $\boldsymbol u$ can be arbitrarily large during transients (it scales with $K_\mathrm{x}$). |
| **No dynamic trade-off** | Cost is instantaneous; problem (3) decouples into (2) (Lemma 13) | No look-ahead or preview, and no trade-off between now and later. |
| **Isolated local minimizers, smooth path** | Assumptions 1.2–1.4: a minimizer exists, minimizers are isolated, $\boldsymbol\chi^\star(t)$ is differentiable | Excludes bifurcations, which do occur (see the jumps in Fig. S3). |
| **Local result** | $\boldsymbol x(0)$ "sufficiently close" to $\boldsymbol\chi^\star(0)$ | Converges to the *nearest* local minimizer. A global basin is conjectured (Proposition 1), not proven. |
| **Exact model, full state** | $\boldsymbol f_\mathrm{A}, \boldsymbol B, \boldsymbol h$ known exactly; $\boldsymbol x$ measured | Robustness to model error, noise and sampling is not analysed. |
| **Continuous time** | Analysis is in continuous time; simulations use RK4 at 1 kHz with $K_\mathrm{x} = 500$ | Stability of the sampled implementation needs $K_\mathrm{x}\,\delta t$ small enough. |

Extensions sketched but not proven: general $\dot{\boldsymbol x} = \boldsymbol f(\boldsymbol x, \boldsymbol u)$ with a right-inverse $\boldsymbol\psi$ (SI 3.7),
and chains of integrators such as robot arms $\ddot{\boldsymbol\theta} = \boldsymbol M^{-1}(\boldsymbol u - \boldsymbol n)$ (SI 8). Section 8 is still fully actuated per degree of freedom.

## 4. What DCC is, in familiar terms

Setting aside the framing, DCC combines three well-known ingredients:

1. **Feedback linearization**, $\boldsymbol u = \boldsymbol B^{-1}(-\boldsymbol f_\mathrm{A} + \boldsymbol v)$. This is possible only because $\boldsymbol B$ is invertible, and it turns the plant into $\dot{\boldsymbol x} = \boldsymbol v$.
2. **Task-priority redundancy resolution** from robotics: a weighted pseudoinverse for the primary task ($\boldsymbol y \to \boldsymbol y_\mathrm{ref}$) plus a null-space projection
   for the secondary objective. The novelty is using the cost Hessian $\boldsymbol R$ as the weight, and a Newton step instead of a gradient step in the null space.
   This is the same structure as resolved-rate control of redundant manipulators, which is Haddadin's home field.
3. A **continuous-time Newton flow** for an equality-constrained problem with a quadratic penalty for the inequalities.
   Optimization-as-dynamics goes back to Arrow–Hurwicz–Uzawa (1958) and Brockett and Helmke & Moore (1990s). More recent work covers
   **time-varying optimization** with prediction–correction (e.g. Fazlyab, Paternain, Preciado & Ribeiro, IEEE TAC 2018, an interior-point
   version; Simonetto et al., Proc. IEEE 2020) and **online feedback optimization** (Hauswirth, Bolognani, Hug & Dörfler, Annual Reviews in Control 2024).

So the result is a well-engineered **tracker of a moving optimum**. Its speed comes from doing **one Newton step per sample**,
warm-started from the current state. The SQP baseline was instead run to convergence with about 290 function evaluations per sample.
A fairer baseline would be the *real-time iteration* scheme (Diehl, Bock & Schlöder, SIAM J. Control Optim. 2005), which also does one SQP step per sample,
or a warm-started prediction–correction method. The per-step cost of DCC is $O(d_\mathrm{x}^3)$ (dense Hessian inverse), the same order as one Newton or SQP iteration.

None of this makes the method wrong. For fully actuated, redundant allocation problems (thrusters, muscles, portfolio weights, robot joints)
it is a clean, cheap and provably convergent controller. But "optimal control without optimization" is better read as
"optimization by feedback, one Newton step at a time".

*References in this section are cited from memory: verify before reuse.*

## 5. Toy example: two-rotor thrust allocation

A 2-state reduction of Example R1 (aerial-fleet control allocation). The states $x_1, x_2$ are rotor speeds (normalized), and the inputs $u_1, u_2$ are motor commands.

- Dynamics: first-order motor lag with quadratic aerodynamic drag, $\dot x_i = -a_i x_i - c_i x_i|x_i| + b_i u_i$. This is control-affine, $\boldsymbol B = \mathrm{diag}(b_i)$, and nonlinear.
- Output: total thrust $y = k_1 x_1^2 + k_2 x_2^2$. This is nonlinear, and $d_\mathrm{y} = 1 < d_\mathrm{x} = 2$ gives one redundant degree of freedom.
- Cost: power-like, $\sigma = \tfrac14\sum_i w_i x_i^4 + \tfrac12\varepsilon\|\boldsymbol x\|^2$, which is strongly convex. Rotor 2 is less efficient ($w_2 > w_1$, $k_2 < k_1$).
- Inequality constraints: $0 \le x_i \le x_{i,\max}$. Rotor 1 is limited to $x_{1,\max} = 1$, which becomes active as thrust demand rises.

Below the limit, the optimal allocation in terms of thrust $z_i = x_i^2$ is $z_i \propto k_i / w_i$. Once rotor 1 saturates, rotor 2 must supply the rest.
The reference ramps up, holds, then steps down.

This example satisfies every assumption in Section 3.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [ ]:
# Plant parameters
a = np.array([1.0, 1.5])  # linear damping
c = np.array([0.5, 0.3])  # quadratic (aerodynamic) drag
b = np.array([2.0, 1.0])  # input gains
k = np.array([1.0, 0.6])  # thrust coefficients
w = np.array([1.0, 2.0])  # cost weights (rotor 2 less efficient)
eps = 0.05  # small quadratic term -> strong convexity at x = 0
x_max = np.array([1.0, 1.5])  # speed limits

# Linear inequality constraints g(x) = G x + c_g <= 0:  x <= x_max, -x <= 0
G = np.vstack([np.eye(2), -np.eye(2)])
c_g = np.concatenate([-x_max, np.zeros(2)])


def f_A(x):
    return -a * x - c * x * np.abs(x)


def B(x):
    return np.diag(b)


def h(t, x):
    return np.array([k @ x**2])


def H(t, x):
    "Output Jacobian dh/dx"
    return (2 * k * x).reshape(1, -1)


def sigma(t, x):
    return 0.25 * w @ x**4 + 0.5 * eps * x @ x


def q(t, x):
    "Gradient of sigma"
    return w * x**3 + eps * x


def Q(t, x):
    "Hessian of sigma"
    return np.diag(3 * w * x**2 + eps)


def g(t, x):
    return G @ x + c_g


def y_ref(t):
    "Ramp up, hold, step down."
    if t < 6:
        return np.array([0.3 + 0.25 * t])
    return np.array([1.8 if t < 8 else 0.6])

In [ ]:
def augmented_grad_hess(t, x, p2):
    "Gradient r and Hessian R of the augmented cost, eq. (S70) (p1 -> infinity)."
    gv = g(t, x)
    active = gv > 0  # violated constraints only: exterior penalty
    Ga = G[active]
    r = q(t, x) + p2**2 * Ga.T @ gv[active]
    R = Q(t, x) + p2**2 * Ga.T @ Ga
    return r, R


def dcc_eta(t, x, p2=100.0):
    "Optimality-emergent direction eta(t, x, y_ref) from eq. (S5), without the gain K_x."
    r, R = augmented_grad_hess(t, x, p2)
    Hx = H(t, x)
    Rinv = np.linalg.inv(R)
    P = Rinv @ Hx.T @ np.linalg.inv(Hx @ Rinv @ Hx.T)  # R-weighted pseudoinverse
    N = np.eye(len(x)) - P @ Hx  # null-space projector
    e_y = y_ref(t) - h(t, x)  # output error
    e_sg = -Rinv @ r  # Newton step on augmented cost
    return P @ e_y + N @ e_sg


def dcc_control(t, x, Kx=20.0, p2=100.0):
    "DCC control law, eq. (4) / (S4a)."
    Binv = np.linalg.inv(B(x))
    return Binv @ (-f_A(x)) + Binv * Kx @ dcc_eta(t, x, p2)


def rk4_step(x, u, dt):
    "One RK4 step of the plant with u held constant (zero-order hold)."
    fx = lambda x: f_A(x) + B(x) @ u
    k1 = fx(x)
    k2 = fx(x + dt / 2 * k1)
    k3 = fx(x + dt / 2 * k2)
    k4 = fx(x + dt * k3)
    return x + dt / 6 * (k1 + 2 * k2 + 2 * k3 + k4)


def simulate(x0, t, Kx=20.0, p2=100.0):
    dt = t[1] - t[0]
    X = np.zeros((len(t), len(x0)))
    U = np.zeros_like(X)
    x = np.array(x0, dtype=float)
    for i, ti in enumerate(t):
        u = dcc_control(ti, x, Kx, p2)
        X[i], U[i] = x, u
        x = rk4_step(x, u, dt)
    Y = np.array([h(ti, xi)[0] for ti, xi in zip(t, X)])
    return X, U, Y

### Offline reference: solve (2) at each instant with SLSQP

This plays the role of the paper's "offline SQP" reference, $\boldsymbol\chi^\star(t)$.

In [ ]:
def solve_static(t, x_guess):
    res = minimize(
        lambda z: sigma(t, z),
        x_guess,
        jac=lambda z: q(t, z),
        constraints=[
            {"type": "eq", "fun": lambda z: h(t, z) - y_ref(t)},
            {"type": "ineq", "fun": lambda z: -g(t, z)},
        ],
        method="SLSQP",
        options={"ftol": 1e-12, "maxiter": 200},
    )
    return res.x, res.nit


dt = 1e-3
t = np.arange(0, 10 + dt / 2, dt)
x0 = np.array([0.6, 0.2])

t_ref = t[::20]
chi = np.zeros((len(t_ref), 2))
nit = np.zeros(len(t_ref), dtype=int)
guess = np.array([0.5, 0.5])
for j, tj in enumerate(t_ref):
    chi[j], nit[j] = solve_static(tj, guess)
    guess = chi[j]  # warm start
print(f"SLSQP iterations per instant (warm-started): mean {nit.mean():.1f}, max {nit.max()}")

In [ ]:
X, U, Y = simulate(x0, t, Kx=20.0, p2=100.0)
Yref = np.array([y_ref(ti)[0] for ti in t])

In [ ]:
def make_io_plot(t, Y, Yref, X, U, t_ref=None, chi=None, x_max=None):
    "Outputs, states, inputs stacked on a common time axis. Returns fig, axes."
    n_plots = 3
    fig, axes = plt.subplots(n_plots, 1, figsize=(7, 1 + 1.5 * n_plots), sharex=True)

    ax = axes[0]
    (line,) = ax.plot(t, Y, label="$y$ (thrust)")
    ax.plot(t, Yref, ":", color=line.get_color(), label=r"$y_\mathrm{ref}$")
    ax.set_ylabel("Thrust")
    ax.grid(True)
    ax.legend()
    ax.set_title("Outputs")

    ax = axes[1]
    for i in range(X.shape[1]):
        (line,) = ax.plot(t, X[:, i], label=f"$x_{i + 1}$")
        if chi is not None:
            ax.plot(t_ref, chi[:, i], ":", color=line.get_color(), lw=2,
                    label=rf"$\chi^\star_{i + 1}$ (offline)")
    if x_max is not None:
        ax.axhline(x_max[0], color="k", lw=0.8, ls="--", label=r"$x_{1,\max}$")
    ax.set_ylim(0, 1.7)
    ax.set_ylabel("Rotor speed")
    ax.grid(True)
    ax.legend(ncol=3, fontsize="small", loc="upper left")
    ax.set_title("States")

    ax = axes[2]
    for i in range(U.shape[1]):
        ax.plot(t, U[:, i], label=f"$u_{i + 1}$")
    ax.set_ylabel("Motor command")
    ax.set_xlabel("Time (s)")
    ax.grid(True)
    ax.legend()
    ax.set_title("Inputs")

    plt.tight_layout()
    return fig, axes


fig, axes = make_io_plot(t, Y, Yref, X, U, t_ref, chi, x_max)
plt.show()

In [ ]:
X_on_ref = X[::20]
err = np.linalg.norm(X_on_ref - chi, axis=1)
ramp = (t_ref > 1) & (t_ref < 6)
print(f"State error |x - chi*| during ramp: max {err[ramp].max():.4f}")
print(f"Max x1 (limit {x_max[0]}): {X[:, 0].max():.5f}")
print(f"Max |u| over run: {np.abs(U).max():.1f}  (no input limits in the formulation)")

**Observations**

- After a fast initial transient ($\sim 1/K_\mathrm{x}$), $\boldsymbol x(t)$ tracks the offline optimum. During the ramp there is a small lag, consistent with the
  $D_\chi/(\alpha K_\mathrm{x})$ bound.
- When rotor 1 reaches its limit, the allocation switches smoothly to rotor 2. The limit is exceeded slightly ($x_1 \approx 1.0003$), as expected for an exterior penalty.
- The input is unconstrained, and the step in $y_\mathrm{ref}$ at $t = 8$ s produces a large input spike that scales with $K_\mathrm{x}$.
  In a real system this would hit actuator limits that DCC does not account for.

## 6. Experiments

### 6a. One Newton step per sample, no plant

Section 4 claims that DCC's speed comes from taking a single warm-started Newton-type step per sample.
To check this, run the *iteration* (S17), $\boldsymbol\chi_{k+1} = \boldsymbol\chi_k + \gamma\,\boldsymbol\eta(t_{k+1}, \boldsymbol\chi_k)$, as a pure optimizer.
There is no plant, and one function evaluation per sample. This is a one-step "real-time iteration" of an optimizer, at a coarse 20 ms sample period.

In [ ]:
def one_step_tracker(chi0, t_samples, gamma=1.0, p2=100.0):
    chi_k = np.array(chi0, dtype=float)
    out = np.zeros((len(t_samples), len(chi0)))
    for j, tj in enumerate(t_samples):
        chi_k = chi_k + gamma * dcc_eta(tj, chi_k, p2)  # one step per sample
        out[j] = chi_k
    return out


chi_1step = one_step_tracker(x0, t_ref)
err_1step = np.linalg.norm(chi_1step - chi, axis=1)
print(f"One step per 20 ms sample: max error during ramp {err_1step[ramp].max():.2e}, "
      f"after t=1 s excluding steps {err_1step[(t_ref > 1) & (np.abs(t_ref - 8) > 0.1)].max():.2e}")

With full steps ($\gamma = 1$), the discrete iteration tracks the moving optimum more tightly than the continuous-time controller, because it is not limited by $K_\mathrm{x}$.
It uses one function evaluation per sample, just like DCC. This supports the reading that the method's efficiency comes from *one warm-started Newton step per sample*,
which other real-time optimizers also exploit, rather than from avoiding optimization altogether.

For comparison, the warm-started SLSQP reference above needed about 11 iterations per instant on average, at a very tight tolerance.
The paper's online SQP baseline used about 290 function evaluations per sample.

### 6b. Effect of the gain $K_\mathrm{x}$

The paper's tracking bound is $\|\boldsymbol x - \boldsymbol\chi^\star\| \lesssim D_\chi/(\alpha K_\mathrm{x})$. Higher gain means less lag but larger inputs.
Between 6 s and 8 s the reference is constant, yet the error levels off near $10^{-3}$ for all gains. That residual is the exterior-penalty offset on the active constraint, not tracking lag (see 6c).

In [ ]:
Kx_values = [5.0, 20.0, 100.0]
results = {Kx: simulate(x0, t, Kx=Kx) for Kx in Kx_values}

n_plots = 2
fig, axes = plt.subplots(n_plots, 1, figsize=(7, 1 + 1.5 * n_plots), sharex=True)

ax = axes[0]
for Kx, (Xk, Uk, Yk) in results.items():
    ax.semilogy(t_ref, np.linalg.norm(Xk[::20] - chi, axis=1), label=f"$K_x$ = {Kx:g}")
ax.set_ylabel(r"$\|x - \chi^\star\|$")
ax.grid(True)
ax.legend()
ax.set_title("State error from offline optimum")

ax = axes[1]
for Kx, (Xk, Uk, Yk) in results.items():
    ax.plot(t, np.linalg.norm(Uk, axis=1), alpha=0.7, label=f"$K_x$ = {Kx:g}")
ax.set_yscale("log")
ax.set_ylabel(r"$\|u\|$")
ax.set_xlabel("Time (s)")
ax.grid(True)
ax.set_title("Inputs")

plt.tight_layout()
plt.show()

### 6c. Effect of the penalty parameter $p_2$

The inequality constraint is enforced by an exterior penalty. The violation shrinks rapidly with $p_2$ (close to $1/p_2^2$ until other errors dominate), but $\boldsymbol R$ becomes increasingly ill-conditioned.

In [ ]:
for p2 in [3.0, 10.0, 100.0, 1000.0]:
    Xp, Up, Yp = simulate(x0, t, Kx=20.0, p2=p2)
    print(f"p2 = {p2:7.1f}:  max x1 - x1_max = {Xp[:, 0].max() - x_max[0]:.2e}")

### 6d. True plant driven by clipped inputs

Here the motor commands (power settings) are limited to $0 \le u_i \le 2.5$. The controller is unchanged: it computes the DCC command from the measured state as before, unaware of the limits.
The true plant receives the clipped command. Nothing in the problem class enforces the lower limit, which means the motors cannot brake: rotors can only slow down by drag.

**Plant states vs model states.** DCC is a static state-feedback law. It doesn't simulate an internal copy of the model; it evaluates $\boldsymbol f_\mathrm A$, $\boldsymbol B$ and $\boldsymbol h$ at the *measured* plant state.
So there is only one state trajectory, and no separate model state to drift. The mismatch shows up instead as a difference between
the rate of change the controller *intends*, $\dot{\boldsymbol x}_\mathrm{int} = \boldsymbol f_\mathrm A(\boldsymbol x) + \boldsymbol B(\boldsymbol x)\boldsymbol u_\mathrm{cmd} = K_\mathrm x\boldsymbol\eta$,
and what actually happens, $\dot{\boldsymbol x} = \boldsymbol f_\mathrm A(\boldsymbol x) + \boldsymbol B(\boldsymbol x)\,\mathrm{clip}(\boldsymbol u_\mathrm{cmd})$.
(With an observer, or with model error, a separate model state would exist and could differ from the plant.)

In [ ]:
def simulate_with_limits(x0, t, Kx=20.0, p2=100.0, u_min=None, u_max=None):
    "DCC with the true plant driven by clipped commands. Returns intended and actual state rates too."
    dt = t[1] - t[0]
    X = np.zeros((len(t), len(x0)))
    U_cmd, U_app, Xdot_int, Xdot_act = (np.zeros_like(X) for _ in range(4))
    x = np.array(x0, dtype=float)
    for i, ti in enumerate(t):
        u_cmd = dcc_control(ti, x, Kx, p2)
        u_app = np.clip(u_cmd, u_min, u_max) if u_min is not None else u_cmd
        X[i], U_cmd[i], U_app[i] = x, u_cmd, u_app
        Xdot_int[i] = f_A(x) + B(x) @ u_cmd  # = Kx * eta
        Xdot_act[i] = f_A(x) + B(x) @ u_app
        x = rk4_step(x, u_app, dt)
    Y = np.array([h(ti, xi)[0] for ti, xi in zip(t, X)])
    return X, U_cmd, U_app, Y, Xdot_int, Xdot_act


u_lims = (np.zeros(2), np.full(2, 2.5))


def settling_time(t, Y, Yref, t_step=8.0, tol=0.01):
    m = t > t_step
    late = np.where(np.abs(Y - Yref)[m] > tol)[0]
    return t[m][late.max()] - t_step if len(late) else 0.0


print(f"{'Kx':>5s} {'limits':>8s} {'settling after 8 s step':>24s} {'time saturated':>15s} {'min u_cmd':>10s}")
for Kx in [20.0, 100.0]:
    for lims in [None, u_lims]:
        Xc, Uc, Ua, Yc, _, _ = simulate_with_limits(x0, t, Kx, 100.0, *(lims or (None, None)))
        sat_time = np.any(Uc != Ua, axis=1).sum() * dt
        print(f"{Kx:5g} {'clipped' if lims else 'none':>8s} {settling_time(t, Yc, Yref):22.3f} s "
              f"{sat_time:13.3f} s {Uc.min():10.1f}")

In [ ]:
Kx = 20.0
X0, U0, _, Y0, _, _ = simulate_with_limits(x0, t, Kx)
Xc, Uc, Ua, Yc, Xd_int, Xd_act = simulate_with_limits(x0, t, Kx, 100.0, *u_lims)

n_plots = 4
fig, axes = plt.subplots(n_plots, 1, figsize=(7, 1 + 1.5 * n_plots), sharex=True)

ax = axes[0]
ax.plot(t, Y0, label="$y$ (no limits)")
ax.plot(t, Yc, label="$y$ (clipped inputs)")
ax.plot(t, Yref, "k:", label=r"$y_\mathrm{ref}$")
ax.set_ylabel("Thrust")
ax.grid(True)
ax.legend(fontsize="small")
ax.set_title("Outputs")

ax = axes[1]
for i, color in enumerate(["C0", "C1"]):
    ax.plot(t, X0[:, i], color=color, alpha=0.4, label=f"$x_{i + 1}$ (no limits)")
    ax.plot(t, Xc[:, i], color=color, label=f"$x_{i + 1}$ (clipped)")
    ax.plot(t_ref, chi[:, i], ":", color=color, lw=2, label=rf"$\chi^\star_{i + 1}$")
ax.set_ylabel("Rotor speed")
ax.grid(True)
ax.legend(ncol=3, fontsize="small")
ax.set_title("States")

ax = axes[2]
for i, color in enumerate(["C0", "C1"]):
    ax.plot(t, Xd_int[:, i], ":", color=color, label=rf"$\dot x_{i + 1}$ intended ($K_x\eta$)")
    ax.plot(t, Xd_act[:, i], color=color, label=rf"$\dot x_{i + 1}$ actual")
ax.set_ylabel("Rate of change")
ax.grid(True)
ax.legend(ncol=2, fontsize="small")
ax.set_title("State rates: controller's intention vs true plant")

ax = axes[3]
for i, color in enumerate(["C0", "C1"]):
    ax.plot(t, Uc[:, i], ":", color=color, label=f"$u_{i + 1}$ commanded")
    ax.plot(t, Ua[:, i], color=color, label=f"$u_{i + 1}$ applied")
for lim in u_lims:
    ax.axhline(lim[0], color="k", lw=0.8, ls="--")
ax.set_ylabel("Motor power")
ax.set_xlabel("Time (s)")
ax.grid(True)
ax.legend(ncol=2, fontsize="small")
ax.set_title("Inputs")

axes[0].set_xlim(7.8, 9.0)  # zoom on the step down at t = 8 s
axes[2].set_ylim(-30, 5)
axes[3].set_ylim(-16, 4)
plt.tight_layout()
plt.show()

**Observations**

- **Upper limit:** it is touched only briefly. Steady operation needs at most about 2.3, so it hardly matters here.
- **Lower limit:** when the reference steps down at 8 s, DCC asks for strongly negative power to brake the rotors. Clipped at zero, the rotors coast down under drag alone, and the output takes several times longer to settle.
- **Higher gain doesn't help under saturation:** raising $K_\mathrm x$ shortens the unclipped response but not the clipped one, and it only makes the commanded inputs larger.
- **Recovery is clean:** as soon as the commands come off the limit, the state rejoins the optimal path. DCC has no integrator, so there is no windup.
- **Direction is distorted, not just speed:** while saturated, each rotor decelerates at its own drag rate rather than along the optimal allocation.
  The plant leaves the optimal path while saturated (states plot), and the actual rates differ from the intended rates $K_\mathrm x\boldsymbol\eta$ (rates plot).
  The orthogonal split between output tracking and cost minimization holds only while no input is saturated, so the convergence proof doesn't cover this period.
- **Conclusion:** with clipped inputs the method still works in practice, with transient errors after large reference changes.
  This is acceptable for slowly varying targets (e.g. a steady-state optimizer), but the paper's guarantees don't apply during saturation.

## 7. Other candidate toy systems
8. Summary of limitations, with page references

Ordered from simplest to most informative:

1. **Two-rotor allocation (above).** Minimal, intuitive, and satisfies all assumptions. Easy to extend to $n$ rotors to check scaling against a warm-started QP or SQP.
2. **Paper's Example M1 (SI 4.1, eq. S75).** Also 2-D, but with a deliberately contrived time-varying cost and constraint set.
   It is useful to reproduce the paper's Fig. S3 and to see local-minimizer jumps, but it is not physically meaningful.
3. **Planar 2-link arm with a 1-D task** (SI 8.3, eq. S124, a chain of two integrators). Example: control only the end-effector $x$ position and minimize joint effort or
   a joint-limit cost. This is the natural home of the method (redundancy resolution) and tests the unproven second-order extension (S118).
4. **A process example where DCC does *not* apply,** e.g. a CSTR with one input and two states (concentration, temperature) and an economic cost.
   $\boldsymbol B$ is not square, so the method cannot be used as stated. Comparing with MPC or real-time optimization shows the boundary of the problem class.
5. **Small portfolio (Example R3 with 3–5 assets).** Mean-variance with a return-equality constraint and box constraints. Here $\boldsymbol h$ is linear and $\sigma$ quadratic,
   so the "system" is an artificial integrator $\dot{\boldsymbol x} = \boldsymbol u$ and DCC reduces to a continuous-time Newton or KKT flow for a time-varying QP.

## 8. Summary of limitations, with page references

Page references: "p." is a main-text page (same as the PDF page). "SI p." is the page number printed in the Supplementary Information; the PDF page is SI page + 34.

### 8.1 Full actuation, not just control affinity

Control-affine only means $\dot{\boldsymbol x} = \boldsymbol f(\boldsymbol x) + \boldsymbol g(\boldsymbol x)\boldsymbol u$, with $\boldsymbol g$ possibly tall. DCC needs the special case where $\boldsymbol g$ is square and invertible.

- Main text, p. 7, below (1b): "$\boldsymbol x \in \mathbb R^{d_\mathrm x}$, $\boldsymbol u \in \mathbb R^{d_\mathrm x}$". The input has the same dimension as the state.
- SI p. 6, below (S1): "The input matrix $\boldsymbol B(\boldsymbol x) \in \mathbb R^{d_\mathrm x \times d_\mathrm x}$ is invertible $\forall \boldsymbol x$". The control law uses $\boldsymbol B(\boldsymbol x)^{-1}$ in (S4b) and (S4c) (SI p. 10).
- Discussion, pp. 21–22, says "control-affinity does not hold in certain systems where the number of inputs is smaller than the number of states". The property that fails there is full actuation, not control affinity.
- The general form $\dot{\boldsymbol x} = \boldsymbol f(\boldsymbol x, \boldsymbol u)$ (SI 3.7, (S73)–(S74), SI p. 40) still needs a right inverse $\boldsymbol\psi$ with $\boldsymbol\ell = \boldsymbol f(\boldsymbol x, \boldsymbol\psi(\boldsymbol x, \boldsymbol\ell))$ for *every* $\boldsymbol\ell \in \mathbb R^{d_\mathrm x}$. Every state velocity must be reachable, which requires $d_u \ge d_x$. The paper leaves "in-depth investigation … for future work".
- The chain-of-integrators extension (SI 8, SI pp. 64–67) keeps $\dim\boldsymbol u = \dim\boldsymbol\theta$, as in a fully actuated robot arm. It is stated without proof (SI p. 65).

### 8.2 No input constraints, input cost or rate limits

- Never stated explicitly; the reader has to infer it. Problems 1 and 2 (SI pp. 7–8) allow any $\boldsymbol u \in \mathbb R^{d_\mathrm x}$. The limitations paragraph of the Discussion (pp. 21–22) lists hybrid systems, control affinity and combinatorial problems, but not input limits.
- In Example R1 the actuator limits ("payload capacity", SI p. 45) are imposed on the *states*, which are modelled as a fast lag $\dot{\boldsymbol x} = -100\boldsymbol x + 100\boldsymbol u$ ((S77), SI p. 43). The commands $\boldsymbol u$ are unbounded, and $K_\mathrm x = 500$ (p. 23).
- In the toy example (Section 5), the reference step produces an input spike that grows with $K_\mathrm x$ (Section 6b).
- Clipping the commands in the true plant (Section 6d) still works in practice, but settling after a step is slower, the state leaves the optimal path while saturated, and the convergence proof does not cover saturation.

### 8.3 The dynamics are not part of the optimization problem

- Problems (2) and (3) (p. 9) contain neither the dynamics (1a) nor the input. The only coupling is (3d), $\boldsymbol x(0) = \boldsymbol\phi(0)$.
- The implicit justification is at the start of SI 3.4 (SI p. 30): any trajectory $\boldsymbol\phi$ can be produced by $\boldsymbol u = \boldsymbol B(\boldsymbol\phi)^{-1}(\dot{\boldsymbol\phi} - \boldsymbol f_\mathrm A(\boldsymbol\phi))$. This relies on $\boldsymbol B$ being invertible and $\boldsymbol u$ being unbounded.
- Lemma 13 (SI p. 32): problem (3) breaks into a sequence of static problems (2), one at each instant. There is no preview, no look-ahead and no trade-off between now and later. "Optimal control" here means tracking a moving static optimum.
- The transient is not optimized. Strictly, $\boldsymbol\phi^\star$ jumps from $\boldsymbol x(0)$ to $\boldsymbol\chi^\star(0^+)$, which would need an infinite input. The claim proven is asymptotic: convergence to within a radius $\propto 1/K_\mathrm x$ (Problems 1 and 2, SI pp. 7–8).

### 8.4 Cost and constraints

- The cost must be **strongly convex** and $C^2$ (Assumption 1.1, SI p. 7), despite "pseudoconvex" in the title and abstract. Non-convexity enters only through a nonlinear equality $\boldsymbol h$.
- Inequality constraints must be **linear** in $\boldsymbol x$: $\boldsymbol g = \boldsymbol G(t)\boldsymbol x + \boldsymbol c(t)$ ((2c), p. 9).
- The "barrier" (S7) (SI p. 13) becomes an **exterior quadratic penalty** when $p_1 \to \infty$ ((S69)–(S70), SI p. 38). It acts only when a constraint is already violated, so constraints are violated slightly, as shown in Section 6c. Choosing $p_2$ is a trade-off with the conditioning of $\boldsymbol R$ ((S72), SI p. 39).
- $\boldsymbol H$ must have full row rank along the trajectory, or the weighted pseudoinverse is singular (for example, at $\boldsymbol x = \boldsymbol 0$ in the toy example).

### 8.5 Convergence is local

- Theorem 1 requires $\boldsymbol x(0)$ to be "sufficiently close" to a local minimizer (SI p. 9; Remark 3, SI p. 28). The state converges to the nearest local minimizer, and minimizer paths can jump (Fig. S3, SI p. 42).
- A global basin of attraction depends on Proposition 1, which is posited without proof: "We leave rigorously proofing Proposition 1 (and consequently, Theorem 4) for future work" (SI pp. 29–30).
- An automated review (Pith, not checked by a person) also questions the validity of the proof of Theorem 1. I have not verified this.

### 8.6 Model, measurement and sampling

- $\boldsymbol f_\mathrm A$, $\boldsymbol B$ and $\boldsymbol h$ are assumed known exactly, and the full state $\boldsymbol x$ is measured. The paper gives no analysis of robustness to model error, noise or sampling. The analysis is in continuous time; the simulations use RK4 at 1 kHz (pp. 23–24).

### 8.7 The "without optimization" claim and the benchmark

- DCC is the continuous-time version of one quasi-Newton (chord) step on the KKT system per time step, iteration (S17) (SI pp. 15–19). It is a warm-started single step, not the absence of optimization.
- The SQP baseline used about 290 function evaluations per sample (p. 14; eq. (5), p. 13). The paper compares against no single-step or warm-started methods, such as real-time iteration or prediction–correction schemes for time-varying optimization. In the toy example, a plain one-step discrete iteration tracks the optimum as well as DCC (Section 6a).
- Cost per step is $O(d_\mathrm x^3)$ for the dense Hessian inverse (eq. (5), p. 13; SI 5). This is the same order as one Newton or SQP iteration.

### 8.8 Possible use: steady-state optimizer

*(Bill's observation)* DCC could serve as a steady-state optimizer, like an RTO layer, that executes much more slowly than the slowest process dynamics. If the inputs can reach the target in a short time compared with the execution period, the fact that inputs are bounded matters little.

*(Claude's comment)* In that setting $\boldsymbol x$ would be the setpoints passed to lower-level controllers. The "system" $\dot{\boldsymbol x} = \boldsymbol u$ is then trivially fully actuated, and $\boldsymbol h$ is the steady-state model. However, at slow execution rates a standard NLP solver run to convergence is affordable too, so DCC's speed advantage would only matter for very large problems. The online feedback optimization literature (Section 4) covers this use case, including robustness to model error.